### Import required libraries


In [ ]:
import os
import warnings
import joblib
from pathlib import Path
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, KFold, GridSearchCV, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder, PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

### Load NYC TLC Dataset


In [ ]:
DATA_DIR = Path("../data/raw")
DATA_DIR.mkdir(parents=True, exist_ok=True)

DATA_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2025-01.parquet"
DATA_PATH = DATA_DIR / "yellow_tripdata_2025-01.parquet"


if not DATA_PATH.exists():
    df = pd.read_parquet(DATA_URL)
    df.to_parquet(DATA_PATH, index=False)

else:
    df = pd.read_parquet(DATA_PATH)

In [3]:
df = df.sample(n=200000, random_state=42)

df.head()

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
1661588,2,2025-01-18 20:53:30,2025-01-18 21:00:47,1.0,0.97,1.0,N,238,166,1,8.6,1.00,0.5,2.22,0.0,1.0,13.32,0.0,0.0,0.00
2269859,1,2025-01-25 11:12:51,2025-01-25 11:17:57,1.0,0.60,1.0,N,50,48,2,5.8,3.25,0.5,0.00,0.0,1.0,10.55,2.5,0.0,0.75
1876274,1,2025-01-21 15:09:31,2025-01-21 15:19:02,1.0,0.80,1.0,N,236,237,1,9.3,2.50,0.5,3.35,0.0,1.0,16.65,2.5,0.0,0.00
976447,2,2025-01-11 22:25:45,2025-01-11 22:34:22,2.0,1.93,1.0,N,231,68,1,10.7,1.00,0.5,3.92,0.0,1.0,20.37,2.5,0.0,0.75
2962359,2,2025-01-04 23:37:07,2025-01-04 23:45:58,NaN,4.44,NaN,None,137,88,0,26.3,0.00,0.5,4.82,0.0,1.0,35.12,NaN,NaN,0.00


### Basic Dataset Overview


In [4]:
print("shape :", df.shape)
df.info()

shape : (200000, 20)
<class 'pandas.core.frame.DataFrame'>
Index: 200000 entries, 1661588 to 655504
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   VendorID               200000 non-null  int32         
 1   tpep_pickup_datetime   200000 non-null  datetime64[us]
 2   tpep_dropoff_datetime  200000 non-null  datetime64[us]
 3   passenger_count        169124 non-null  float64       
 4   trip_distance          200000 non-null  float64       
 5   RatecodeID             169124 non-null  float64       
 6   store_and_fwd_flag     169124 non-null  object        
 7   PULocationID           200000 non-null  int32         
 8   DOLocationID           200000 non-null  int32         
 9   payment_type           200000 non-null  int64         
 10  fare_amount            200000 non-null  float64       
 11  extra                  200000 non-null  float64       
 12  mta_tax               

In [5]:
df.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
VendorID,200000.0,NaN,NaN,NaN,1.786685,1.0,2.0,2.0,2.0,7.0,0.425221
tpep_pickup_datetime,200000,NaN,NaN,NaN,2025-01-17 11:11:26.663485,2024-12-31 21:15:22,2025-01-10 07:23:20,2025-01-17 15:41:16,2025-01-24 20:16:12.500000,2025-01-31 23:59:51,NaN
tpep_dropoff_datetime,200000,NaN,NaN,NaN,2025-01-17 11:26:28.497899,2024-12-31 21:26:00,2025-01-10 07:41:43.500000,2025-01-17 15:58:31,2025-01-24 20:28:45.750000,2025-02-01 18:54:29,NaN
passenger_count,169124.0,NaN,NaN,NaN,1.297941,0.0,1.0,1.0,1.0,6.0,0.751262
trip_distance,200000.0,NaN,NaN,NaN,4.78547,0.0,0.98,1.66,3.09,92185.93,365.910719
RatecodeID,169124.0,NaN,NaN,NaN,2.449303,1.0,1.0,1.0,1.0,99.0,11.494701
store_and_fwd_flag,169124,2,N,168732,NaN,NaN,NaN,NaN,NaN,NaN,NaN
PULocationID,200000.0,NaN,NaN,NaN,165.305265,1.0,132.0,162.0,234.0,265.0,64.534735
DOLocationID,200000.0,NaN,NaN,NaN,164.06483,1.0,113.0,162.0,234.0,265.0,69.410517
payment_type,200000.0,NaN,NaN,NaN,1.038955,0.0,1.0,1.0,1.0,4.0,0.70374


### Data Understanding

_Check Columns_

In [6]:
df.columns

Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

_Target Column_

In [7]:
x = df.drop(columns=["fare_amount"])
y = df["fare_amount"]

_Separate Numerical and Categorical Columns_

In [8]:
num_cols = x.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_cols = x.select_dtypes(include=["object"]).columns.tolist()

print("Numerical columns:", num_cols)
print("Categorical columns:", cat_cols)

Numerical columns: ['passenger_count', 'trip_distance', 'RatecodeID', 'payment_type', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee']
Categorical columns: ['store_and_fwd_flag']


### Data Quality Check

_Missing Values_

In [9]:
missing_df = df.isnull().sum().sort_values(ascending=True)

missing_df[missing_df > 0]

,0
store_and_fwd_flag,30876
RatecodeID,30876
passenger_count,30876
congestion_surcharge,30876
Airport_fee,30876


_Duplicate Rows_

In [10]:
duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


_Unique Values in Categorical Columns_

In [11]:
for col in cat_cols:
    print(f"{col}: {df[col].nunique()} unique values")
    print(df[col].value_counts().head())

store_and_fwd_flag: 2 unique values
store_and_fwd_flag
N    168732
Y       392
Name: count, dtype: int64
